In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df1 = pd.read_csv("../../datasets/inverse_repairs.csv")

df1

In [ ]:
len(df1[(df1['constraint Deleted'] == True)] )

In [ ]:
len(df1[(df1['constraint Deprecated'] == True)] )

In [ ]:
len(df1[(df1['Included as Exception'] == True)] )

In [ ]:
len(df1[(df1['T-box requested_property replacement'] == True)] )

In [ ]:
len(df1[(df1['A-box wdt statement Deleted'] == True)] )

In [ ]:
df1['Inverse Added'].value_counts()

In [ ]:
df1.loc[df1['object'].str.startswith('genid'), 'Inverse Added'] = False

In [ ]:
df1['Inverse Added'].value_counts()

In [ ]:
df1['Inverse Added'] = df1['Inverse Added'].astype(str).str.lower().replace({'false': False, 'true': True})

In [ ]:
df1['Inverse Added'].value_counts()

In [ ]:
df1['Inverse Added'] = df1['Inverse Added'].astype(bool)

In [ ]:
len(df1[(df1['Inverse Added'] == True)] )

In [ ]:
df1.dtypes

In [ ]:
df1[(df1['constraint Deleted'] == False) & 
     (df1['constraint Deprecated'] == False)& 
     (df1['Included as Exception'] == False)& 
     (df1['T-box requested_property replacement'] == False)& 
     (df1['A-box wdt statement Deleted'] == False)& 
     (df1['Inverse Added'] == False)
    ]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q42034555"
property = "http://www.wikidata.org/entity/P629"
print(isRemoved(subject, property))


In [ ]:
bkp = df1

In [ ]:
len(bkp[(bkp['constraint Deleted'] == False) & 
     (bkp['constraint Deprecated'] == False)& 
     (bkp['Included as Exception'] == False)& 
     (bkp['T-box requested_property replacement'] == False)& 
     (bkp['A-box wdt statement Deleted'] == False)& 
     (bkp['Inverse Added'] == False)
    ])

In [ ]:
df1[(df1['constraint Deleted'] == False) & 
     (df1['constraint Deprecated'] == False)& 
     (df1['Included as Exception'] == False)& 
     (df1['T-box requested_property replacement'] == False)& 
     (df1['A-box wdt statement Deleted'] == False)& 
     (df1['Inverse Added'] == False)
    ]

In [ ]:
for index, row in df1.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    
    
    if (index % 50000 == 0):
        print(index)
    #break
    
    if row['constraint Deleted'] == False and (row['constraint Deprecated'] == False) and (row['Included as Exception'] == False) and      (row['T-box requested_property replacement'] == False) and (row['A-box wdt statement Deleted'] == False) and    (row['Inverse Added'] == False):
        subject = row['subject']
        property = row['property']
        
        # Call isRemoved function
        removed = isRemoved(subject, property)

        # Update instanceRemoved column
        df1.at[index, 'A-box wdt statement Deleted'] = removed

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
df1

In [ ]:
df1[(df1['constraint Deleted'] == False) & 
     (df1['constraint Deprecated'] == False)& 
     (df1['Included as Exception'] == False)& 
     (df1['T-box requested_property replacement'] == False)& 
     (df1['A-box wdt statement Deleted'] == False)& 
     (df1['Inverse Added'] == False)
    ]

In [ ]:
rows_to_drop = df1[
    (df1['constraint Deleted'] == False) &
    (df1['constraint Deprecated'] == False) &
    (df1['Included as Exception'] == False) &
    (df1['T-box requested_property replacement'] == False) &
    (df1['A-box wdt statement Deleted'] == False) &
    (df1['Inverse Added'] == False) &
    (df1['object'].str.startswith('genid'))
]

In [ ]:
rows_to_drop

In [ ]:
df1_filtered = df1.drop(rows_to_drop.index)

In [ ]:
df1_filtered

In [ ]:
df1_filtered[(df1_filtered['constraint Deleted'] == False) & 
     (df1_filtered['constraint Deprecated'] == False)& 
     (df1_filtered['Included as Exception'] == False)& 
     (df1_filtered['T-box requested_property replacement'] == False)& 
     (df1_filtered['A-box wdt statement Deleted'] == False)& 
     (df1_filtered['Inverse Added'] == False)
    ]

In [ ]:
df1_filtered.to_csv('final_inverse_repairs.csv', index=False)

In [ ]:
len(df1_filtered)

In [ ]:
test = merged_df

In [ ]:
test = pd.merge(test, filtered_df, on=['subject','property','object'], how='left')

In [ ]:
test = test.drop(columns=['deleted constraint_y','deprecated rank_y', 'exception_y', 'requested_property_y'])

In [ ]:
test

In [ ]:
new_column_names = {'deleted constraint_x': 'deleted constraint','deprecated rank_x': 'deprecated rank', 'exception_x': 'exception', 'requested_property_x': 'requested_property'}

In [ ]:
test = test.rename(columns=new_column_names)

In [ ]:
test

In [ ]:
test['instanceRemoved'].unique()

In [ ]:
test = test.fillna(False)

In [ ]:
test

In [ ]:
len(test[(test['deprecated rank'] == False) 
         & (test['deleted constraint'] == True) 
         & (test['instanceRemoved'] == False)
        ])

In [ ]:
len(test[(test['deprecated rank'] == True) 
         & (test['deleted constraint'] == False) 
         & (test['instanceRemoved'] == False)
        ])

In [ ]:
len(test[(test['deprecated rank'] == False) 
         & (test['deleted constraint'] == False) 
         & (test['exception'] == True)
         & (test['instanceRemoved'] == False)
        ])

In [ ]:
len(test[(test['deprecated rank'] == False) 
         & (test['deleted constraint'] == False) 
         & (test['exception'] == False)
         & (test['instanceRemoved'] == True)
        ])

In [ ]:
test[(test['deleted constraint'] == False) & 
     (test['deprecated rank'] == False) & 
     (test['exception'] == False) &
     (test['instanceRemoved'] == False)
    ]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def inverseAdded(obj, req_prop):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    req_prop = req_prop.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{obj}> <{req_prop}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
#test
inverseAdded('http://www.wikidata.org/entity/Q927337','http://www.wikidata.org/prop/direct/P925')

In [ ]:
test.loc[:, 'inverseAdded'] = False

In [ ]:
test_bkp = test

In [ ]:
test

In [ ]:
for index, row in test.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (
     (row['deleted constraint'] == False) & 
     (row['deprecated rank'] == False) & 
     (row['exception'] == False) &
     (row['instanceRemoved'] == False) &
     (row['object'].startswith('http'))
    ):
        obj = row['object']
        requested_property = row['requested_property']

        if (index % 10000 == 0):
            print(index)
        #break

        # Call inverseAdded function
        inverse = inverseAdded(obj, requested_property)

        # Update instanceRemoved column
        test.at[index, 'inverseAdded'] = inverse

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
for index, row in test.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (
     (row['deleted constraint'] == False) & 
     (row['deprecated rank'] == False) & 
     (row['exception'] == False) &
     (row['instanceRemoved'] == False)
    ):
        obj = row['object']
        if(obj.startswith('genid')):
            print(row)

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
len(test[(test['inverseAdded'] == True)])

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedWithObj(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
#subject = "http://www.wikidata.org/entity/Q11861442"
#property = "http://www.wikidata.org/prop/direct/P1026"
#print(isRemoved(subject, property))


In [ ]:
count = 1
for index, row in test.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    if (
     (row['deleted constraint'] == False) & 
     (row['deprecated rank'] == False) & 
     (row['exception'] == False) &
     (row['instanceRemoved'] == False) &
     (row['inverseAdded'] == False) &
     (row['object'].startswith('http'))
    ):
        obj = row['object']
        
        if (count % 10000 == 0):
            print(count)
        #break
        count += 1
        # Call inverseAdded function
        wdtStmtRemoved = isRemovedWithObj(row['subject'], row['property'], obj)

        # Update instanceRemoved column
        test.at[index, 'instanceRemoved'] = wdtStmtRemoved
            

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
len(df1)

In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df1 = pd.read_csv("final_inverse_repairs.csv")

df1

In [ ]:
df1.dtypes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = df1['A-box wdt statement Deleted'] | df1['Inverse Added']
df2['T-box changes'] = (
    df1['constraint Deleted'] | 
    df1['constraint Deprecated'] | 
    df1['Included as Exception'] | 
    df1['T-box requested_property replacement']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("Inverse Constraint share of repairs")

plt.show()
